# LifeMate — hynt Zipformer 153M, chạy đủ 991 audio
Checkpoint `hynt/ZipFormer-150M-CR-CTC-RNNT-6000h`, revision đã khóa.
Kaggle private → bật Internet → thêm dataset LifeMate và output audit-follow-up như cũ → Run All.
CPU 2 threads, FP32 ONNX, greedy; không cần GPU hoặc HF_TOKEN. Chỉ chạy hynt, không chia stage.
Giữ đúng 991 ID/reference cũ, không đưa reference vào inference; có progress và ZIP kết quả.
Kết quả đã có: 991/991 OK, WER 15,34%, CER 10,65%. Không cần chạy lại chỉ vì dọn thư mục.
Notebook mới dùng output riêng; RESUME_ZIP để None khi chạy mới, không trộn config ZIP cũ.
Card ghi CC-BY-NC-ND-4.0; lựa chọn inference chưa xác nhận quyền fine-tune/triển khai.


In [ ]:
import sys, subprocess
print('Chuẩn bị thư viện đọc input và uv...', flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    '--upgrade-strategy', 'only-if-needed', 'uv', 'numpy', 'pandas', 'soundfile==0.13.1', 'huggingface-hub==1.24.0'])
DEPENDENCY_CHECK = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
print(DEPENDENCY_CHECK.stdout + DEPENDENCY_CHECK.stderr)

## 1. Cấu hình
Một checkpoint hynt, đủ 991 mẫu; giữ nguyên revision và manifest.


In [ ]:
from pathlib import Path
DATASET_ROOT = Path('/kaggle/input/datasets/jamel27/zalo-aichallange-lifemate-dataset')
FOLLOWUP_INPUT = Path('/kaggle/input/notebooks/jamel27/zalo-lifemate-audit-follow-up')
CACHE_DIR = None
DATASET_ZIP = None
LLM_PARAMETERS = 4_000_000_000  # SỬA theo checkpoint LLM thực tế, không chỉ theo tên "4B".
OTHER_PARAMETERS = 0          # SỬA nếu có model embedding/reranker/TTS... trong hệ thống.
RETRY_ERRORS = False          # True để thử lại các mẫu lỗi trong cùng config.
RESUME_ZIP = None             # Ví dụ Path('/kaggle/input/.../lifemate_asr_hynt_zipformer_full_test_outputs.zip')
WORK_DIR = Path('/kaggle/working/lifemate_asr_hynt_zipformer_inputs')
OUTPUT_DIR = Path('/kaggle/working/lifemate_asr_hynt_zipformer')
CODE_DIR = Path('/kaggle/working/lifemate_asr_hynt_zipformer_code')
MODELS = [{'name': 'zipformer_hynt_153m', 'family': 'sherpa', 'model_id': 'hynt/ZipFormer-150M-CR-CTC-RNNT-6000h', 'revision': '8c3d474d3d74cb247f4b8c7b0f8b48598ac8c1c6', 'gated': False, 'encoder': 'encoder-epoch-11-avg-2.onnx', 'decoder': 'decoder-epoch-11-avg-2.onnx', 'joiner': 'joiner-epoch-11-avg-2.onnx', 'tokens': 'config.json', 'files': ['encoder-epoch-11-avg-2.onnx', 'decoder-epoch-11-avg-2.onnx', 'joiner-epoch-11-avg-2.onnx', 'config.json']}]


## 2. Mã dùng chung
Đã nhúng sẵn, có thể thu gọn cell; không cần sửa.


In [ ]:
import sys, json, hashlib, shutil, subprocess, traceback, zipfile, os
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import display, FileLink
CODE_DIR.mkdir(parents=True,exist_ok=True)
SOURCES = {'asr_benchmark.py': '"""LifeMate cached ASR benchmark. No dataset crawl, labels or external audio API."""\nfrom pathlib import Path\nimport json, hashlib, os, re, unicodedata, time, platform, threading, traceback, gc, shutil\nimport numpy as np\nimport pandas as pd\n\nNORMALIZATION = \'nfc_lower_punct_symbol_to_space_keep_accents_digits_v1\'\nCODE_VERSION = \'lifemate-model-selection-2.0\'\n\ndef normalize(text):\n    text = unicodedata.normalize(\'NFC\', str(text)).lower()\n    text = \'\'.join(\' \' if unicodedata.category(c)[0] in \'PS\' else c for c in text)\n    return \' \'.join(text.split())\n\ndef digest_file(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(1024*1024), b\'\'): h.update(block)\n    return h.hexdigest()\n\ndef fingerprint(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()\n\ndef safe_path(path):\n    p = Path(path)\n    if any(part.endswith(\'_environment\') for part in p.parts):\n        raise ValueError(\'Environment access forbidden: \'+str(p))\n    return p\n\ndef write_json(path, obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str));tmp.replace(path)\n\ndef append_json(path,obj):\n    with open(path,\'a\',encoding=\'utf-8\') as f:\n        f.write(json.dumps(obj,ensure_ascii=False,default=str,allow_nan=False)+\'\\n\');f.flush();os.fsync(f.fileno())\n\ndef read_journal(path):\n    """Recover only a torn final line; malformed interior records stop the run."""\n    path=Path(path)\n    if not path.exists(): return []\n    raw=path.read_bytes();lines=raw.splitlines(keepends=True);rows=[];offset=0\n    for i,line in enumerate(lines):\n        try: row=json.loads(line)\n        except (json.JSONDecodeError,UnicodeDecodeError):\n            if i!=len(lines)-1: raise RuntimeError(\'Corrupt journal interior: \'+str(path))\n            backup=path.with_name(path.name+\'.torn-\'+str(time.time_ns()))\n            backup.write_bytes(raw)\n            path.write_bytes(raw[:offset])\n            append_json(path.parent/\'recovery.jsonl\',{\'journal\':path.name,\'backup\':backup.name,\'discarded_bytes\':len(raw)-offset})\n            break\n        rows.append(row);offset+=len(line)\n    # A valid final JSON without newline must be terminated before appending.\n    if rows and path.read_bytes() and not path.read_bytes().endswith(b\'\\n\'):\n        with open(path,\'ab\') as f: f.write(b\'\\n\');f.flush();os.fsync(f.fileno())\n    return rows\n\ndef resolve_followup_cache(input_root,working_dir,override=None):\n    """Read the named follow-up mount only; never fall back to old audit metadata."""\n    import zipfile\n    root=safe_path(Path(input_root).resolve())\n    names=[\'references_enriched.csv\',\'split_manifest.csv\',\'benchmark_manifest.csv\']\n    candidates=[safe_path(Path(override).resolve())] if override is not None else [root/\'lifemate_asr_followup\',root]\n    for p in candidates:\n        safe_path(p)\n        if all((p/n).is_file() for n in names):\n            return p,dict(source=str(p),mode=\'csv_directory\')\n    if override is not None:\n        raise FileNotFoundError(\'CACHE_DIR must directly contain the three CSVs: \'+str(override))\n    archive=root/\'lifemate_asr_followup.zip\'\n    if not archive.is_file():\n        raise FileNotFoundError(\'Follow-up cache missing. Add the saved notebook output at \'+str(root)+\n            \'; expected lifemate_asr_followup/{\'+\',\'.join(names)+\'} or lifemate_asr_followup.zip.\')\n    with zipfile.ZipFile(archive) as z:\n        members={}\n        for info in z.infolist():\n            parts=info.filename.replace(\'\\\\\',\'/\').split(\'/\')\n            if info.is_dir() or \'__MACOSX\' in parts or any(x.endswith(\'_environment\') for x in parts): continue\n            if parts[-1] in names:\n                if any(x in [\'..\',\'\'] for x in parts) or info.filename.startswith(\'/\'):\n                    raise ValueError(\'Unsafe cache ZIP member\')\n                if parts[-1] in members: raise ValueError(\'Ambiguous cache CSV in ZIP: \'+parts[-1])\n                members[parts[-1]]=info\n        if set(members)!=set(names): raise FileNotFoundError(\'Follow-up ZIP must contain one of each required CSV\')\n        dest=safe_path(Path(working_dir).resolve()/\'followup_cache\');dest.mkdir(parents=True,exist_ok=True)\n        for name,info in members.items():\n            target=dest/name\n            if target.is_symlink(): raise ValueError(\'Symlink cache target forbidden\')\n            target.write_bytes(z.read(info))\n    return dest,dict(source=str(archive),mode=\'three_csvs_from_zip\')\n\n\ndef prepare_selected_audio(rows,dataset_root,working_dir,archive_path=None):\n    """Use selected files or extract only their exact entries from the dataset ZIP."""\n    import zipfile\n    root=safe_path(Path(dataset_root).resolve())\n    if all(resolve_audio(x,root).is_file() for x in rows.path):\n        return root,dict(source=str(root),mode=\'mounted_files\',n_selected=len(rows))\n    archives=[safe_path(Path(archive_path).resolve())] if archive_path is not None else [\n        p for p in [root/\'train+public_test.zip\',root/\'data.zip\'] if p.is_file()]\n    if len(archives)!=1:\n        raise FileNotFoundError(\'Selected audio missing under \'+str(root)+\n            \'. Mount extracted train+public_test/ or set DATASET_ZIP to the exact ZIP path.\')\n    archive=archives[0]\n    dest=safe_path(Path(working_dir).resolve()/\'selected_audio\');dest.mkdir(parents=True,exist_ok=True)\n    requested={resolve_audio(x,dest).relative_to(dest).as_posix() for x in rows.path}\n    with zipfile.ZipFile(archive) as z:\n        members={}\n        for info in z.infolist():\n            parts=info.filename.replace(\'\\\\\',\'/\').split(\'/\')\n            if info.is_dir() or \'__MACOSX\' in parts or any(x.endswith(\'_environment\') for x in parts): continue\n            if info.filename not in requested: continue\n            if info.filename in members: raise ValueError(\'Duplicate selected audio entry in ZIP\')\n            members[info.filename]=info\n        if set(members)!=requested:\n            raise FileNotFoundError(f\'ZIP missing {len(requested-set(members))} selected audio entries; expected train+public_test/... paths\')\n        required=sum(i.file_size for i in members.values())\n        if shutil.disk_usage(dest).free < required+16*1024*1024: raise OSError(\'Not enough space for selected audio\')\n        for name,info in members.items():\n            target=resolve_audio(name,dest)\n            target.parent.mkdir(parents=True,exist_ok=True)\n            tmp=target.with_suffix(target.suffix+\'.tmp\')\n            if tmp.is_symlink(): raise ValueError(\'Symlink extraction target forbidden\')\n            try:\n                with z.open(info) as src,open(tmp,\'wb\') as out: shutil.copyfileobj(src,out)\n                tmp.replace(target)\n            finally:\n                if tmp.exists(): tmp.unlink()\n    return dest,dict(source=str(archive),mode=\'selected_zip_entries\',n_selected=len(rows),bytes_selected=required)\n\n\ndef load_cache(cache_dir):\n    p=safe_path(cache_dir)\n    names=[\'references_enriched.csv\',\'split_manifest.csv\',\'benchmark_manifest.csv\']\n    for n in names:\n        if not (p/n).is_file(): raise FileNotFoundError(f\'Missing cache {p/n}; set CACHE_DIR to the directory containing the three CSV files.\')\n    r,s,b=[pd.read_csv(p/n,keep_default_na=False) for n in names]\n    for d in [r,s,b]:\n        if d.ref_uid.duplicated().any(): raise ValueError(\'Duplicate ref_uid\')\n    if not set(s.partition)<= {\'development\',\'validation\'}: raise ValueError(\'Unexpected partition\')\n    j=s[[\'ref_uid\',\'partition\',\'group_id\']].merge(r,on=\'ref_uid\',validate=\'one_to_one\',how=\'left\',indicator=True)\n    if not j._merge.eq(\'both\').all(): raise ValueError(\'Split IDs missing from reference cache\')\n    if not j.split.eq(\'train\').all() or not j.manifest.map(lambda x:Path(x).name==\'train.jsonl\').all():\n        raise ValueError(\'Public test or non-train manifest in labeled evaluation\')\n    # Compare by ID, never infer pairing by row order.\n    by=r.set_index(\'ref_uid\')\n    for d in [s,b]:\n        if not set(d.ref_uid)<=set(by.index): raise ValueError(\'Unmapped ID\')\n        for col in [\'path\',\'reference\',\'sample_uid\',\'kind\']:\n            if col in d and not (d[col].to_numpy()==by.loc[d.ref_uid,col].to_numpy()).all():\n                raise ValueError(\'Cache disagreement: \'+col)\n    dev=j[j.partition==\'development\'];val=j[j.partition==\'validation\']\n    if set(dev.sample_uid)&set(val.sample_uid) or set(dev.group_id)&set(val.group_id): raise ValueError(\'Group leakage\')\n    if not set(b.ref_uid)<=set(val.ref_uid): raise ValueError(\'Benchmark not validation-only\')\n    if len(r)!=6969 or len(s)!=5036 or len(val)!=991 or len(b)!=100: raise ValueError(\'Unexpected cache version; inspect explicitly before changing expected counts\')\n    if val.reference.map(lambda x:not isinstance(x,str) or not x.strip()).any(): raise ValueError(\'Missing gold text\')\n    for x in r.path: safe_path(x)\n    hashes={n:digest_file(p/n) for n in names}\n    return r,j.drop(columns=\'_merge\'),b,hashes\n\ndef select_rows(joined,benchmark,stage):\n    if stage==\'validation\': rows=joined[joined.partition==\'validation\'].copy().sort_values(\'ref_uid\')\n    elif stage in [\'smoke\',\'benchmark\']:\n        rows=joined.set_index(\'ref_uid\').loc[benchmark.ref_uid].reset_index()\n        if stage==\'smoke\':\n            # Deterministic endpoints per kind exercise short/long final/history; do not reshuffle benchmark.\n            picks=[]\n            for kind in [\'final\',\'history\']:\n                sub=rows[rows.kind==kind].sort_values([\'duration\',\'ref_uid\'])\n                picks+=sub.head(2).ref_uid.tolist()+sub.tail(2).ref_uid.tolist()\n            rows=rows[rows.ref_uid.isin(picks)].copy()\n    else: raise ValueError(\'stage must be smoke / benchmark / validation\')\n    return rows.reset_index(drop=True)\n\ndef resolve_audio(old_path,dataset_root):\n    old=safe_path(old_path)\n    parts=old.parts\n    if \'train+public_test\' not in parts: raise ValueError(\'No known root marker: \'+str(old))\n    tail=Path(*parts[parts.index(\'train+public_test\'):])\n    root=safe_path(dataset_root).resolve()\n    target=safe_path(root/tail).resolve()\n    if not target.is_relative_to(root): raise ValueError(\'Audio escaped configured root\')\n    safe_path(target)\n    return target\n\ndef runtime_info():\n    import torch,transformers,soundfile,psutil,importlib.metadata as md\n    versions={n:md.version(n) for n in [\'torch\',\'transformers\',\'numpy\',\'pandas\',\'soundfile\',\'psutil\',\'huggingface-hub\',\'tokenizers\',\'safetensors\']}\n    return dict(versions=versions,python=platform.python_version(),platform=platform.platform(),cpu=platform.processor(),\n                cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n                gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,\n                cpu_threads=torch.get_num_threads())\n\ndef resolve_checkpoint(spec,out,offline=False):\n    """Pin remote branch to SHA once; local mounted checkpoint gets content fingerprint."""\n    from huggingface_hub import HfApi,snapshot_download\n    lock=out/(spec[\'name\']+\'_checkpoint.json\')\n    old=json.loads(lock.read_text()) if lock.exists() else None\n    if spec.get(\'local_path\'):\n        source=safe_path(spec[\'local_path\']).resolve()\n        if not source.is_dir(): raise FileNotFoundError(\'Checkpoint mount missing: \'+str(source))\n        if not (source/\'config.json\').is_file(): raise ValueError(\'Local checkpoint must contain config.json; no scan performed\')\n        entries=[]\n        # Only checkpoint files, never dataset audio or environment directories.\n        for root,dirs,files in os.walk(source):\n            dirs[:]=sorted(d for d in dirs if not d.endswith(\'_environment\') and d not in [\'.git\',\'.cache\'])\n            for name in sorted(files):\n                f=safe_path((Path(root)/name).resolve())\n                if f.suffix in [\'.json\',\'.bin\',\'.safetensors\',\'.txt\',\'.model\',\'.jinja\']:\n                    entries.append((str(f.relative_to(source)),digest_file(f)))\n        if not entries or not (source/\'config.json\').exists(): raise ValueError(\'Incomplete local checkpoint\')\n        identity=\'local-sha256:\'+fingerprint(entries)\n        obj=dict(model_id=spec[\'model_id\'],revision=identity,requested_revision=spec.get(\'revision\',\'main\'),local_files=entries)\n    else:\n        if offline and not old: raise ValueError(\'OFFLINE requires local_path or a previously saved checkpoint lock and HF cache\')\n        sha=old[\'revision\'] if old and old.get(\'model_id\')==spec[\'model_id\'] and old.get(\'requested_revision\')==spec.get(\'revision\',\'main\') else None\n        if offline and sha is None: raise ValueError(\'Checkpoint lock does not match offline request\')\n        if sha is None: sha=HfApi().model_info(spec[\'model_id\'],revision=spec.get(\'revision\',\'main\')).sha\n        source=Path(snapshot_download(spec[\'model_id\'],revision=sha,local_files_only=offline,\n                allow_patterns=[\'*.json\',\'*.txt\',\'*.model\',\'*.jinja\',\'*.safetensors\',\'pytorch_model*.bin\']))\n        identity=sha;obj=dict(model_id=spec[\'model_id\'],revision=sha,requested_revision=spec.get(\'revision\',\'main\'))\n    write_json(lock,obj)\n    return source,identity\n\nclass RSSMonitor:\n    """Sample process RSS every 10 ms; brief peaks may be missed; not system RAM."""\n    def __enter__(self):\n        import psutil\n        self.process=psutil.Process();self.peak=self.process.memory_info().rss;self.stop=threading.Event()\n        def sample():\n            while not self.stop.wait(.01): self.peak=max(self.peak,self.process.memory_info().rss)\n        self.thread=threading.Thread(target=sample,daemon=True);self.thread.start();return self\n    def __exit__(self,*args):\n        self.peak=max(self.peak,self.process.memory_info().rss);self.stop.set();self.thread.join()\n\ndef whisper_sequence_diagnostics(sequence,eos_token_id,pad_token_id=None):\n    """Check raw generate sequences, not Whisper\'s EOS-stripped segment tokens."""\n    tokens=sequence.tolist() if hasattr(sequence,\'tolist\') else list(sequence)\n    eos=set(eos_token_id if isinstance(eos_token_id,(list,tuple,set)) else [eos_token_id])\n    if None in eos or not eos: raise ValueError(\'Whisper EOS token is not configured\')\n    # Some generation configurations use a separate pad token after EOS.\n    while tokens and tokens[-1]==pad_token_id and pad_token_id not in eos: tokens.pop()\n    return dict(output_token_count=len(tokens),ended_with_eos=bool(tokens and tokens[-1] in eos))\n\n\nclass ASR:\n    def __init__(self,spec,source,revision,device=\'cuda:0\',precision=\'float16\',max_new_tokens=400):\n        import torch\n        self.spec=spec;self.revision=revision;self.device=device;self.family=spec[\'family\'];self.max_new_tokens=max_new_tokens\n        self.dtype=torch.float16 if precision==\'float16\' else torch.float32\n        if device.startswith(\'cuda\') and not torch.cuda.is_available(): raise RuntimeError(\'Requested CUDA unavailable; select Kaggle GPU or explicitly set DEVICE=cpu and PRECISION=float32\')\n        if device==\'cpu\' and precision!=\'float32\': raise ValueError(\'Use float32 on CPU\')\n        if self.family==\'seq2seq\':\n            from transformers import WhisperProcessor as proc,WhisperForConditionalGeneration as model\n        elif self.family==\'ctc\':\n            from transformers import Wav2Vec2Processor as proc,Wav2Vec2ForCTC as model\n        elif self.family==\'qwen3_asr\':\n            from transformers import AutoProcessor as proc,Qwen3ASRForConditionalGeneration as model\n        else: raise ValueError(\'Unsupported model family: \'+str(self.family))\n        self.processor=proc.from_pretrained(str(source),local_files_only=True)\n        self.model=model.from_pretrained(str(source),local_files_only=True,dtype=self.dtype,attn_implementation=\'eager\').to(device).eval()\n        if self.family==\'seq2seq\': self.model.generation_config.forced_decoder_ids=None\n    def sync(self):\n        import torch\n        if self.device.startswith(\'cuda\'): torch.cuda.synchronize(self.device)\n    def transcribe(self,audio_path):\n        """Only audio enters inference. No reference, context, tool schema or confidence."""\n        import torch,soundfile as sf\n        result=dict(status=\'error\',text_model=\'\',text_normalized=\'\',model_id=self.spec[\'model_id\'],revision=self.revision,\n                    latency_ms=None,model_ms=None,duration_s=None,rtf=None,warnings=[],error_code=None)\n        self.sync();start=time.perf_counter();mon=None\n        try:\n            if self.device.startswith(\'cuda\'): torch.cuda.reset_peak_memory_stats(self.device)\n            with RSSMonitor() as mon:\n                wav,sr=sf.read(safe_path(audio_path),dtype=\'float32\',always_2d=True)\n                if sr!=16000 or wav.shape[1]!=1: raise ValueError(\'UNEXPECTED_AUDIO_FORMAT: expected mono 16000Hz\')\n                wav=wav[:,0];duration=len(wav)/sr;result[\'duration_s\']=duration\n                if not len(wav) or not np.isfinite(wav).all(): raise ValueError(\'INVALID_AUDIO\')\n                if self.family==\'seq2seq\' and duration>30: raise ValueError(\'LONG_AUDIO_UNSUPPORTED: no truncation permitted\')\n                if self.family==\'ctc\' and duration>10: result[\'warnings\'].append(\'above_model_card_10s_recommendation_full_audio_used\')\n                if self.family==\'seq2seq\':\n                    batch=self.processor(wav,sampling_rate=sr,return_tensors=\'pt\',return_attention_mask=True,truncation=False)\n                    inputs={k:v.to(self.device,dtype=self.dtype) if k==\'input_features\' else v.to(self.device) for k,v in batch.items()}\n                elif self.family==\'ctc\':\n                    batch=self.processor(wav,sampling_rate=sr,return_tensors=\'pt\',padding=False)\n                    inputs={k:v.to(self.device,dtype=self.dtype) if k==\'input_values\' else v.to(self.device) for k,v in batch.items()}\n                elif self.family==\'qwen3_asr\':\n                    # Native HF processor accepts the waveform already validated above.\n                    # Language is fixed globally; no transcript/context/hotwords are passed.\n                    # Pinned native processor defaults to 16 kHz and PyTorch tensors.\n                    batch=self.processor.apply_transcription_request(audio=wav,language=\'vi\')\n                    inputs={k:v.to(device=self.device,dtype=self.dtype) if v.is_floating_point() else v.to(self.device) for k,v in batch.items()}\n                self.sync();model_start=time.perf_counter()\n                with torch.inference_mode():\n                    if self.family==\'seq2seq\':\n                        # HF Whisper\'s default tensor output strips EOS during segment processing.\n                        # The structured short-form output retains raw sequences including EOS.\n                        generated=self.model.generate(**inputs,language=\'vi\',task=\'transcribe\',num_beams=1,do_sample=False,\n                                                max_new_tokens=self.max_new_tokens,return_timestamps=False,\n                                                return_dict_in_generate=True,output_scores=False,output_logits=False)\n                        ids=generated.sequences\n                    elif self.family==\'ctc\': ids=self.model(**inputs).logits.argmax(dim=-1)\n                    else:\n                        generated=self.model.generate(**inputs,max_new_tokens=self.max_new_tokens,num_beams=1,do_sample=False,\n                            return_dict_in_generate=True,output_scores=False,output_logits=False)\n                        ids=generated.sequences[:,inputs[\'input_ids\'].shape[1]:]\n                self.sync();result[\'model_ms\']=(time.perf_counter()-model_start)*1000\n                if self.family==\'qwen3_asr\':\n                    raw=self.processor.decode(ids,return_format=\'raw\')\n                    parsed=self.processor.decode(ids,return_format=\'transcription_only\')\n                    result[\'text_output_raw\']=raw[0] if isinstance(raw,list) else raw\n                    text=parsed[0] if isinstance(parsed,list) else parsed\n                    if \'<asr_text>\' not in result[\'text_output_raw\']: raise RuntimeError(\'QWEN_OUTPUT_SCHEMA_MISSING_ASR_TEXT\')\n                else: text=self.processor.batch_decode(ids,skip_special_tokens=True)[0]\n                result[\'text_model\']=text;result[\'text_normalized\']=normalize(text)\n                if self.family in [\'seq2seq\',\'qwen3_asr\']:\n                    diagnostics=whisper_sequence_diagnostics(ids[0],self.model.generation_config.eos_token_id,\n                        self.model.generation_config.pad_token_id)\n                    result.update(diagnostics)\n                    # Keep partial text for diagnosis, but never score a genuinely unfinished sequence as success.\n                    if not diagnostics[\'ended_with_eos\']: raise RuntimeError(\'TOKEN_LIMIT_OR_MISSING_EOS\')\n                if not text.strip(): raise RuntimeError(\'EMPTY_TRANSCRIPT\')\n                result[\'status\']=\'ok\'\n        except Exception as exc:\n            result[\'error_code\']=type(exc).__name__+\': \'+str(exc)\n        finally:\n            self.sync();result[\'latency_ms\']=(time.perf_counter()-start)*1000\n            if result[\'duration_s\']: result[\'rtf\']=result[\'latency_ms\']/1000/result[\'duration_s\']\n            result[\'rss_peak_bytes\']=mon.peak if mon else None\n            result[\'vram_peak_allocated_bytes\']=torch.cuda.max_memory_allocated(self.device) if self.device.startswith(\'cuda\') else None\n            result[\'vram_peak_reserved_bytes\']=torch.cuda.max_memory_reserved(self.device) if self.device.startswith(\'cuda\') else None\n        return result\n\ndef run_benchmark(rows,specs,cfg,cache_hashes):\n    import torch\n    out=safe_path(cfg[\'output_dir\']);out.mkdir(parents=True,exist_ok=True)\n    env=runtime_info();write_json(out/\'runtime.json\',env)\n    torch.manual_seed(2026);np.random.seed(2026)\n    # preflight only selected audio paths, no decoding/hash/scan; fail clearly on missing mounts.\n    if not safe_path(cfg[\'dataset_root\']).is_dir(): raise FileNotFoundError(\'DATASET_ROOT missing: \'+str(cfg[\'dataset_root\']))\n    paths={x.ref_uid:str(resolve_audio(x.path,cfg[\'dataset_root\'])) for x in rows.itertuples()}\n    print(\'Selected audio paths missing:\',sum(not Path(p).is_file() for p in paths.values()),\'(each is recorded as a sample error)\')\n    write_json(out/\'selection.json\',dict(stage=cfg[\'stage\'],ref_uids=rows.ref_uid.tolist(),cache_hashes=cache_hashes))\n    run_dirs=[]\n    for spec in specs:\n        engine=None\n        try:\n            source,revision=resolve_checkpoint(spec,out,cfg.get(\'offline\',False))\n            token_limit=spec.get(\'max_new_tokens\',cfg[\'max_new_tokens\'])\n            identity=dict(code=CODE_VERSION,source_sha256=cfg[\'source_sha256\'],model_id=spec[\'model_id\'],revision=revision,\n                          family=spec[\'family\'],device=cfg[\'device\'],precision=cfg[\'precision\'],max_new_tokens=token_limit,\n                          decoding=\'greedy_vi_no_reference_no_context_no_external_LM\',normalization=NORMALIZATION,cache_hashes=cache_hashes,\n                          runtime=env)\n            run_id=fingerprint(identity)[:20];rd=out/(spec[\'name\']+\'_\'+run_id);rd.mkdir(exist_ok=True);run_dirs.append(rd)\n            write_json(rd/\'config.json\',identity);journal=rd/\'predictions.jsonl\'\n            prev=read_journal(journal)\n            latest={x[\'ref_uid\']:x for x in prev}\n            completed={uid for uid,x in latest.items() if x[\'status\']==\'ok\' or not cfg.get(\'retry_errors\',False)}\n            pending=rows[~rows.ref_uid.isin(completed)]\n            if pending.empty: print(spec[\'name\'],\'already completed\');continue\n            with RSSMonitor() as load_mon:\n                start=time.perf_counter();engine=ASR(spec,source,revision,cfg[\'device\'],cfg[\'precision\'],token_limit);engine.sync()\n                load_ms=(time.perf_counter()-start)*1000\n            write_json(rd/\'model_config.json\',engine.model.config.to_dict())\n            if getattr(engine.model,\'generation_config\',None) is not None:\n                write_json(rd/\'generation_config.json\',engine.model.generation_config.to_dict())\n            session_id=str(time.time_ns())\n            write_json(rd/(\'session_\'+session_id+\'.json\'),dict(session_id=session_id,load_ms=load_ms,load_rss_peak_bytes=load_mon.peak,hardware=env))\n            warm=engine.transcribe(paths[pending.iloc[0].ref_uid]);write_json(rd/(\'warmup_\'+session_id+\'.json\'),warm)\n            if warm[\'status\']!=\'ok\': print(\'Warmup failed (retained):\',warm[\'error_code\'])\n            for i,row in enumerate(pending.itertuples()):\n                record=engine.transcribe(paths[row.ref_uid])\n                if record.get(\'duration_s\') is not None and abs(record[\'duration_s\']-float(row.duration)) > 1/16000:\n                    record[\'status\']=\'error\';record[\'error_code\']=\'CACHE_AUDIO_DURATION_MISMATCH\'\n                record.update(ref_uid=row.ref_uid,session_id=session_id,run_id=run_id)\n                append_json(journal,record)\n                if i%10==0: print(spec[\'name\'],i+1,\'/\',len(pending),record[\'status\'])\n                if record[\'status\']!=\'ok\' and \'out of memory\' in str(record[\'error_code\']).lower(): torch.cuda.empty_cache()\n        except Exception:\n            err=dict(model=spec[\'name\'],stage=cfg[\'stage\'],traceback=traceback.format_exc(),time_ns=time.time_ns())\n            append_json(out/\'model_errors.jsonl\',err);print(\'MODEL ERROR:\',spec[\'name\'],err[\'traceback\'])\n        finally:\n            del engine;gc.collect()\n            if torch.cuda.is_available(): torch.cuda.empty_cache()\n    return run_dirs\n\ndef edit_distance(a,b):\n    prev=list(range(len(b)+1))\n    for i,x in enumerate(a,1):\n        cur=[i]\n        for j,y in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))\n        prev=cur\n    return prev[-1]\n\ndef score_rows(rows,records):\n    latest={x[\'ref_uid\']:x for x in records};out=[]\n    for row in rows.itertuples():\n        p=latest.get(row.ref_uid);status=p[\'status\'] if p else \'missing\'\n        ref=normalize(row.reference);hyp=normalize(p[\'text_model\']) if p and status==\'ok\' else \'\'\n        # Failure-as-empty keeps every selected reference in the denominator; also report success-only.\n        rc=ref.replace(\' \',\'\');hc=hyp.replace(\' \',\'\')\n        out.append(dict(ref_uid=row.ref_uid,sample_uid=row.sample_uid,kind=row.kind,duration=float(row.duration),status=status,\n                        reference=row.reference,text_model=p.get(\'text_model\',\'\') if p else \'\',word_edits=edit_distance(ref.split(),hyp.split()),\n                        n_words=len(ref.split()),char_edits=edit_distance(rc,hc),n_chars=len(rc),\n                        latency_ms=p.get(\'latency_ms\') if p else None,rtf=p.get(\'rtf\') if p else None,\n                        rss_peak_bytes=p.get(\'rss_peak_bytes\') if p else None,\n                        vram_peak_allocated_bytes=p.get(\'vram_peak_allocated_bytes\') if p else None,\n                        vram_peak_reserved_bytes=p.get(\'vram_peak_reserved_bytes\') if p else None,\n                        warnings=json.dumps(p.get(\'warnings\',[])) if p else \'[]\'))\n    return pd.DataFrame(out)\n\ndef aggregate(d):\n    ok=d[d.status==\'ok\'];timed=ok.dropna(subset=[\'latency_ms\'])\n    return dict(n=len(d),n_ok=len(ok),n_error=int(d.status.eq(\'error\').sum()),n_missing=int(d.status.eq(\'missing\').sum()),\n        complete=bool(not d.status.eq(\'missing\').any()),failure_rate=float(d.status.ne(\'ok\').mean()),\n        wer=float(d.word_edits.sum()/d.n_words.sum()) if len(ok) and d.n_words.sum() else None,\n        cer_no_spaces=float(d.char_edits.sum()/d.n_chars.sum()) if len(ok) and d.n_chars.sum() else None,\n        wer_success_only=float(ok.word_edits.sum()/ok.n_words.sum()) if ok.n_words.sum() else None,\n        cer_success_only=float(ok.char_edits.sum()/ok.n_chars.sum()) if ok.n_chars.sum() else None,\n        latency_p50_ms=float(timed.latency_ms.median()) if len(timed) else None,\n        latency_p95_ms=float(timed.latency_ms.quantile(.95)) if len(timed) else None,\n        rtf_weighted=float((timed.latency_ms.sum()/1000)/timed.duration.sum()) if len(timed) else None,\n        rss_peak_bytes=int(ok.rss_peak_bytes.max()) if ok.rss_peak_bytes.notna().any() else None,\n        vram_peak_allocated_bytes=int(ok.vram_peak_allocated_bytes.max()) if ok.vram_peak_allocated_bytes.notna().any() else None,\n        vram_peak_reserved_bytes=int(ok.vram_peak_reserved_bytes.max()) if ok.vram_peak_reserved_bytes.notna().any() else None)\n\ndef evaluate(rows,out_dir,run_dirs=None):\n    out=Path(out_dir);summaries=[];scored={}\n    # Enumerate only immediate benchmark output children, never dataset.\n    for rd in sorted(out.iterdir() if run_dirs is None else set(map(Path,run_dirs))):\n        if not rd.is_dir() or not (rd/\'config.json\').is_file(): continue\n        d=score_rows(rows,read_journal(rd/\'predictions.jsonl\'));d.to_csv(rd/\'per_audio.csv\',index=False);scored[rd.name]=d\n        groups={\'all\':d,\'final\':d[d.kind==\'final\'],\'history\':d[d.kind==\'history\'],\n                \'duration_le10s\':d[d.duration<=10],\'duration_gt10s\':d[d.duration>10]}\n        # Human annotations are read only AFTER inference; hints are not gold categories.\n        ann=out/\'dialect_annotations.csv\'\n        if ann.exists():\n            labels=pd.read_csv(ann,keep_default_na=False)\n            if labels.ref_uid.duplicated().any(): raise ValueError(\'Duplicate annotation IDs; adjudicate first\')\n            # Annotation templates also carry duration/kind/sample_uid for human review.\n            # Preserve scoring metadata from the validated rows when joining labels.\n            merged=d.merge(labels,on=\'ref_uid\',how=\'left\',validate=\'one_to_one\',suffixes=(\'\', \'_annotation\'))\n            for col in [\'accent_region\',\'accent_variation\',\'dialect_vocabulary\',\'vi_accented_english\',\'code_switching\',\'noise\']:\n                if col in merged:\n                    merged[col]=merged[col].fillna(\'unannotated\').replace(\'\', \'unannotated\')\n                    for label,sub in merged.groupby(col,dropna=False): groups[f\'{col}={label}\']=sub\n        for name,sub in groups.items():\n            if len(sub): summaries.append(dict(run=rd.name,subset=name,**aggregate(sub)))\n    summary=pd.DataFrame(summaries);summary.to_csv(out/\'metrics.csv\',index=False)\n    return summary,scored\n\n\ndef selection_table(summary,scored,out_path,max_latency_p95_ms=None,max_vram_gb=None):\n    """Describe eligible measured tradeoffs; never declare a winner without human review."""\n    records=[]\n    if not summary.empty:\n        for row in summary[summary.subset==\'all\'].to_dict(\'records\'):\n            eligible=row[\'complete\'] and row[\'n_ok\']==row[\'n\'] and row[\'n\']>0 and pd.notna(row[\'wer\'])\n            reasons=[]\n            if not eligible: reasons.append(\'incomplete_or_errors\')\n            if max_latency_p95_ms is not None:\n                if pd.isna(row[\'latency_p95_ms\']) or row[\'latency_p95_ms\']>max_latency_p95_ms:\n                    eligible=False;reasons.append(\'latency_limit_or_missing_measurement\')\n            if max_vram_gb is not None:\n                peak=row[\'vram_peak_reserved_bytes\']\n                if pd.isna(peak) or peak>max_vram_gb*1024**3:\n                    eligible=False;reasons.append(\'vram_limit_or_missing_measurement\')\n            records.append(dict(**row,eligible=bool(eligible),excluded_reason=\';\'.join(reasons),human_review_required=True))\n    result=pd.DataFrame(records)\n    if not result.empty:\n        result[\'pareto_wer_latency\']=False\n        eligible=result[result.eligible & result.latency_p95_ms.notna()]\n        for i,a in eligible.iterrows():\n            dominated=any((other.wer<=a.wer and other.latency_p95_ms<=a.latency_p95_ms and\n                (other.wer<a.wer or other.latency_p95_ms<a.latency_p95_ms)) for _,other in eligible.iterrows())\n            result.loc[i,\'pareto_wer_latency\']=not dominated\n        result=result.sort_values([\'eligible\',\'wer\',\'latency_p95_ms\'],ascending=[False,True,True])\n    result.to_csv(out_path,index=False)\n    return result\n\ndef paired_bootstrap(scored,out_path,n_boot=2000,seed=2026):\n    """Cluster paired bootstrap by conversation, conditional on fixed evaluation subset."""\n    rng=np.random.default_rng(seed);names=sorted(scored);results=[]\n    for i,a in enumerate(names):\n        for b in names[i+1:]:\n            x,y=scored[a],scored[b]\n            if not set(x.ref_uid)==set(y.ref_uid) or x.status.eq(\'missing\').any() or y.status.eq(\'missing\').any(): continue\n            if not x.status.eq(\'ok\').any() or not y.status.eq(\'ok\').any(): continue\n            merged=x[[\'ref_uid\',\'sample_uid\',\'word_edits\',\'n_words\']].merge(y[[\'ref_uid\',\'word_edits\']],on=\'ref_uid\',suffixes=(\'_a\',\'_b\'),validate=\'one_to_one\')\n            groups=merged.groupby(\'sample_uid\')[[\'word_edits_a\',\'word_edits_b\',\'n_words\']].sum().to_numpy()\n            if len(groups)<2: continue\n            deltas=[]\n            for _ in range(n_boot):\n                sampled=groups[rng.integers(0,len(groups),len(groups))].sum(axis=0)\n                deltas.append((sampled[0]-sampled[1])/sampled[2])\n            results.append(dict(a=a,b=b,n_conversations=len(groups),delta_wer_a_minus_b=float((groups[:,0].sum()-groups[:,1].sum())/groups[:,2].sum()),\n                                ci95_low=float(np.quantile(deltas,.025)),ci95_high=float(np.quantile(deltas,.975)),n_boot=n_boot))\n    pd.DataFrame(results,columns=[\'a\',\'b\',\'n_conversations\',\'delta_wer_a_minus_b\',\'ci95_low\',\'ci95_high\',\'n_boot\']).to_csv(out_path,index=False)\n    return results\n\ndef save_template_preserving_rows(new,path):\n    if path.exists():\n        previous=pd.read_csv(path,keep_default_na=False)\n        if previous.ref_uid.duplicated().any(): raise ValueError(\'Duplicate annotation IDs\')\n        new=pd.concat([previous,new[~new.ref_uid.isin(previous.ref_uid)]],ignore_index=True)\n    new.to_csv(path,index=False)\n\ndef annotation_templates(rows,out_dir,seed=2026):\n    out=Path(out_dir)\n    ann=rows[[\'ref_uid\',\'sample_uid\',\'kind\',\'path\',\'duration\']].copy()\n    ann[\'selection_reason\']=\'fixed_benchmark\' if len(rows)<=100 else \'fixed_validation\'\n    for col in [\'annotator_id\',\'accent_region\',\'accent_variation\',\'dialect_vocabulary\',\'vi_accented_english\',\'code_switching\',\'noise\',\'reference_problem\',\'notes\']:\n        ann[col]=\'\'\n    save_template_preserving_rows(ann, out/\'dialect_annotations.csv\')\n    gold=rows[[\'ref_uid\',\'reference\']].copy()\n    for cat in [\'person\',\'money\',\'time\',\'location\',\'device\',\'negation\']: gold[cat+\'_spans_json\']=\'\'\n    save_template_preserving_rows(gold, out/\'critical_gold_annotation.csv\')\n\ndef make_review(scored,out_dir):\n    records=[]\n    for name,d in scored.items():\n        for row in d.itertuples():\n            r=dict(model=name,ref_uid=row.ref_uid,status=row.status,reference=row.reference,hypothesis=row.text_model)\n            for cat in [\'person\',\'money\',\'time\',\'location\',\'device\',\'negation\']:r[cat+\'_judgment\']=\'\'\n            r[\'notes\']=\'\';records.append(r)\n    # Never overwrite filled human review.\n    path=Path(out_dir)/\'critical_model_review_template.csv\'\n    new=pd.DataFrame(records,columns=[\'model\',\'ref_uid\',\'status\',\'reference\',\'hypothesis\',\n        \'person_judgment\',\'money_judgment\',\'time_judgment\',\'location_judgment\',\'device_judgment\',\'negation_judgment\',\'notes\'])\n    if path.exists():\n        old=pd.read_csv(path,keep_default_na=False)\n        keys=[\'model\',\'ref_uid\']\n        if old.duplicated(keys).any(): raise ValueError(\'Duplicate model review IDs\')\n        # Preserve the reviewed hypothesis and judgments, including after retry.\n        # A changed hypothesis needs explicit human review rather than silent replacement.\n        seen=set(map(tuple,old[keys].to_numpy()))\n        new=pd.concat([old,new[[tuple(k) not in seen for k in new[keys].to_numpy()]]],ignore_index=True)\n    new.to_csv(path,index=False)\n\ndef package_outputs(out_dir,zip_base):\n    """Export comparison artifacts only; standalone and usable after interrupted runs."""\n    from pathlib import Path\n    import hashlib, json, zipfile\n    root=Path(out_dir).resolve()\n    if not root.is_dir(): raise FileNotFoundError(root)\n    target=Path(str(zip_base)+\'.zip\').resolve()\n    target.parent.mkdir(parents=True,exist_ok=True)\n    shared={\'runtime.json\',\'selection.json\',\'benchmark_request.json\',\'benchmark_source.py\',\'input_provenance.json\',\n            \'scoring_patch.json\',\'scoring_source.py\',\n            \'metrics.csv\',\'paired_bootstrap.csv\',\'comparison_status.csv\',\'model_selection.csv\',\'dependency_check.txt\',\n            \'dialect_annotations.csv\',\'critical_gold_annotation.csv\',\n            \'critical_model_review_template.csv\',\'model_errors.jsonl\'}\n    run_files={\'config.json\',\'model_config.json\',\'generation_config.json\',\n               \'predictions.jsonl\',\'per_audio.csv\',\'recovery.jsonl\'}\n    selected=[]\n    for p in sorted(root.iterdir()):\n        if p.is_symlink(): continue\n        if p.is_file() and (p.name in shared or\n                (p.name.startswith(\'selection_\') and p.suffix==\'.csv\') or\n                p.name.endswith(\'_checkpoint.json\')):\n            selected.append(p)\n        elif p.is_dir() and not p.name.endswith(\'_environment\') and (p/\'config.json\').is_file():\n            for f in sorted(p.iterdir()):\n                if f.is_symlink() or not f.is_file(): continue\n                if f.name in run_files or (f.name.startswith((\'session_\',\'warmup_\')) and f.suffix==\'.json\') or f.name.startswith(\'predictions.jsonl.torn-\'):\n                    selected.append(f)\n    entries=[]\n    tmp=target.with_suffix(\'.zip.tmp\')\n    try:\n        with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n            for f in selected:\n                relative=f.relative_to(root).as_posix()\n                data=f.read_bytes()\n                entries.append(dict(path=relative,bytes=len(data),sha256=hashlib.sha256(data).hexdigest()))\n                archive.writestr(relative,data)\n            manifest=dict(schema_version=1,private=True,files=entries,\n                missing_summary_files=[n for n in [\'benchmark_request.json\',\'metrics.csv\',\'comparison_status.csv\'] if not (root/n).is_file()],\n                note=\'May contain partial runs or load failures. Check comparison_status.csv, n_ok/n_missing and model_errors.jsonl. Contains private transcripts; no original audio or weights exported.\')\n            archive.writestr(\'EXPORT_MANIFEST.json\',json.dumps(manifest,ensure_ascii=False,indent=2))\n        tmp.replace(target)\n    finally:\n        if tmp.exists(): tmp.unlink()\n    return str(target)\n', 'parallel_benchmark_report.py': '"""Report both notebooks with identical scoring and explicit incomplete-run status."""\nfrom pathlib import Path\nimport json\nimport zipfile\n\nimport pandas as pd\nimport asr_benchmark as b\n\n\ndef summarize(rows, request):\n    root = Path(request[\'output_dir\'])/request[\'stage\']\n    root.mkdir(parents=True, exist_ok=True)\n    summaries, all_rows, errors = [], [], []\n    for spec in request[\'models\']:\n        directory = root/spec[\'name\']\n        journal = directory/\'predictions.jsonl\'\n        records = b.read_journal(journal)\n        ids = set(rows.ref_uid)\n        if any(x[\'ref_uid\'] not in ids for x in records):\n            raise ValueError(\'Predictions from a different selection: \'+str(journal))\n        config = directory/\'config.json\'\n        if config.exists():\n            saved = json.loads(config.read_text(encoding=\'utf-8\'))\n            if saved[\'input_fingerprint\'] != request[\'input_fingerprint\']:\n                raise ValueError(\'Input fingerprint differs; use a separate OUTPUT_DIR\')\n        elif records:\n            raise ValueError(\'Predictions without config cannot be audited\')\n        latest = {x[\'ref_uid\']: x for x in records}\n        d = b.score_rows(rows, records)\n        d[\'model\'] = spec[\'name\']\n        d[\'audio_path\'] = rows.path.to_numpy()\n        d[\'reference_normalized\'] = rows.reference.map(b.normalize).to_numpy()\n        d[\'hypothesis_normalized\'] = [b.normalize(latest[x][\'text_model\'])\n                                     if x in latest and latest[x][\'status\']==\'ok\' else \'\' for x in d.ref_uid]\n        d[\'exact_normalized\'] = d.status.eq(\'ok\') & d.word_edits.eq(0)\n        d[\'exact_raw\'] = [latest[x].get(\'text_model\') == r if x in latest and latest[x][\'status\']==\'ok\' else False\n                          for x, r in zip(d.ref_uid, d.reference)]\n        d[\'error_code\'] = [latest[x].get(\'error_code\') or \'\' if x in latest else \'NOT_RUN\' for x in d.ref_uid]\n        state_path = directory/\'state.json\'\n        state = json.loads(state_path.read_text(encoding=\'utf-8\')) if state_path.exists() else {\'state\':\'not_started\'}\n        param_path = directory/\'parameters.json\'\n        params = json.loads(param_path.read_text(encoding=\'utf-8\')) if param_path.exists() else {}\n        groups = {\'all\':d, \'final\':d[d.kind==\'final\'], \'history\':d[d.kind==\'history\'],\n                  \'duration_le10s\':d[d.duration<=10], \'duration_gt10s\':d[d.duration>10]}\n        for subset, part in groups.items():\n            if part.empty:\n                continue\n            stats = b.aggregate(part)\n            ready = stats[\'n_missing\']==0 and stats[\'n_ok\']==stats[\'n\'] and state[\'state\']==\'completed\'\n            summaries.append(dict(model=spec[\'name\'], model_id=spec[\'model_id\'], subset=subset,\n                stage=request[\'stage\'], input_fingerprint=request[\'input_fingerprint\'],\n                state=state[\'state\'], comparison_ready=ready, **stats,\n                WER_pct=100*stats[\'wer\'] if stats[\'wer\'] is not None else None,\n                CER_pct=100*stats[\'cer_no_spaces\'] if stats[\'cer_no_spaces\'] is not None else None,\n                exact_normalized=int(part.exact_normalized.sum()),\n                exact_normalized_pct=100*part.exact_normalized.mean(), exact_raw=int(part.exact_raw.sum()),\n                word_edits=int(part.word_edits.sum()), n_words=int(part.n_words.sum()),\n                char_edits=int(part.char_edits.sum()), n_chars=int(part.n_chars.sum()),\n                asr_parameters=params.get(\'asr_parameters\'),system_parameters=params.get(\'system_parameters\'),\n                below_6B=params.get(\'below_6B\')))\n        directory.mkdir(exist_ok=True)\n        d.to_csv(directory/\'per_audio.csv\',index=False,encoding=\'utf-8-sig\')\n        all_rows.append(d)\n        errors.append(d[d.status!=\'ok\'])\n    summary = pd.DataFrame(summaries)\n    summary.to_csv(root/\'metrics.csv\',index=False,encoding=\'utf-8-sig\')\n    per_audio = pd.concat(all_rows, ignore_index=True)\n    per_audio.to_csv(root/\'per_audio_all_models.csv\',index=False,encoding=\'utf-8-sig\')\n    failed = pd.concat(errors, ignore_index=True)\n    failed.to_csv(root/\'errors_and_missing.csv\',index=False,encoding=\'utf-8-sig\')\n    failed.groupby([\'model\',\'status\',\'error_code\'],dropna=False).size().rename(\'count\').reset_index().to_csv(\n        root/\'error_counts.csv\',index=False,encoding=\'utf-8-sig\')\n    return summary\n\n\ndef export_results(request):\n    root = Path(request[\'output_dir\'])/request[\'stage\']\n    target = root.parent/(root.parent.name+\'_\'+request[\'stage\']+\'_outputs.zip\')\n    allowed = {\'.csv\',\'.json\',\'.jsonl\',\'.txt\',\'.py\',\'.yaml\'}\n    entries = []\n    with zipfile.ZipFile(target,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(root.rglob(\'*\')):\n            if p.is_file() and not p.is_symlink() and p.suffix in allowed:\n                relative = p.relative_to(root).as_posix()\n                entries.append(dict(path=relative,sha256=b.digest_file(p),bytes=p.stat().st_size))\n                z.write(p,relative)\n        z.writestr(\'EXPORT_MANIFEST.json\',json.dumps(dict(private=True, files=entries,\n            input_fingerprint=request[\'input_fingerprint\'], stage=request[\'stage\'],\n            normalization=b.NORMALIZATION, inference_executed_by_author=False,\n            note=\'Contains private references/predictions. No audio, model weights, or environments.\'),indent=2))\n    return target\n', 'additional_asr_worker.py': '"""Local-only Ichigo/Sherpa inference for the fixed LifeMate 991-audio selection."""\nfrom pathlib import Path\nimport argparse\nimport importlib.metadata as metadata\nimport json\nimport math\nimport time\nimport traceback\n\nimport numpy as np\nimport asr_benchmark as b\n\nLIMIT = 2_000_000_000\n\n\ndef require_under_limit(count):\n    if isinstance(count, bool) or not isinstance(count, int) or count <= 0 or count >= LIMIT:\n        raise ValueError(f\'ASR parameter count must be strictly below {LIMIT}; got {count}\')\n\n\ndef count_modules(modules):\n    """Count shared parameters once; include floating buffers conservatively."""\n    seen, parameters, buffers = set(), 0, 0\n    for module in modules:\n        for kind, values in [(\'parameters\', module.parameters()), (\'buffers\', module.buffers())]:\n            for value in values:\n                if kind == \'buffers\' and not value.is_floating_point():\n                    continue\n                key = (str(value.device), value.data_ptr())\n                if key in seen:\n                    continue\n                seen.add(key)\n                if kind == \'parameters\':\n                    parameters += value.numel()\n                else:\n                    buffers += value.numel()\n    return dict(parameters=parameters, floating_buffers=buffers, bound=parameters+buffers,\n                basis=\'unique_tensor_parameters_plus_floating_buffers_conservative_bound\')\n\n\ndef onnx_tensor_elements(graph):\n    """Count stored dense tensors, including constants and nested subgraphs."""\n    import onnx\n    if graph.sparse_initializer:\n        raise ValueError(\'Sparse ONNX tensors need explicit parameter accounting\')\n    tensors = list(graph.initializer)\n    total = 0\n    for node in graph.node:\n        for attr in node.attribute:\n            if attr.type == onnx.AttributeProto.TENSOR:\n                tensors.append(attr.t)\n            elif attr.type == onnx.AttributeProto.TENSORS:\n                tensors.extend(attr.tensors)\n            elif attr.type == onnx.AttributeProto.GRAPH:\n                total += onnx_tensor_elements(attr.g)\n            elif attr.type == onnx.AttributeProto.GRAPHS:\n                total += sum(onnx_tensor_elements(g) for g in attr.graphs)\n            elif attr.type in (onnx.AttributeProto.SPARSE_TENSOR, onnx.AttributeProto.SPARSE_TENSORS):\n                raise ValueError(\'Sparse ONNX tensor attributes need explicit accounting\')\n    for tensor in tensors:\n        if tensor.external_data:\n            raise ValueError(\'External ONNX tensor data requires an explicit file manifest\')\n        total += math.prod(tensor.dims)\n    return total\n\n\nclass Engine:\n    uses_cuda = False\n\n    def sync(self):\n        if self.uses_cuda:\n            import torch\n            torch.cuda.synchronize()\n\n    def transcribe(self, path):\n        import soundfile as sf\n        result = dict(status=\'error\', text_model=\'\', error_code=None, warnings=[])\n        self.sync()\n        if self.uses_cuda:\n            import torch\n            torch.cuda.reset_peak_memory_stats()\n        start = time.perf_counter()\n        with b.RSSMonitor() as monitor:\n            try:\n                audio, sr = sf.read(path, dtype=\'float32\', always_2d=True)\n                if sr != 16000 or audio.shape[1] != 1 or not len(audio) or not np.isfinite(audio).all():\n                    raise ValueError(\'Expected nonempty finite mono 16000 Hz audio\')\n                result[\'duration_s\'] = len(audio)/sr\n                if result[\'duration_s\'] > 30:\n                    raise ValueError(\'Audio >30 seconds; no silent truncation\')\n                text = self.decode(audio[:, 0], sr)\n                if not isinstance(text, str) or not text.strip():\n                    raise ValueError(\'Empty/invalid transcript\')\n                result.update(status=\'ok\', text_model=text)\n            except Exception as exc:\n                result[\'error_code\'] = type(exc).__name__+\': \'+str(exc)\n            finally:\n                self.sync()\n                result[\'latency_ms\'] = (time.perf_counter()-start)*1000\n                result[\'rss_peak_bytes\'] = monitor.peak\n                result[\'rtf\'] = result[\'latency_ms\']/1000/result[\'duration_s\'] if result.get(\'duration_s\') else None\n                result[\'vram_peak_allocated_bytes\'] = torch.cuda.max_memory_allocated() if self.uses_cuda else None\n                result[\'vram_peak_reserved_bytes\'] = torch.cuda.max_memory_reserved() if self.uses_cuda else None\n        return result\n\n\nclass Ichigo(Engine):\n    uses_cuda = True\n\n    def __init__(self, paths):\n        import torch\n        import whisper\n        from ichigo_whisper.demo.utils import load_model\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Enable Kaggle GPU\')\n        torch.manual_seed(2026)\n        self.model = load_model(ref=str(paths[\'merge-medium-vi-2d-2560c-dim64.pth\']),\n                                size=\'merge-medium-vi-2d-2560c-dim64\')\n        self.model.ensure_whisper(\'cuda:0\', language=\'vi\')\n        self.model.to(\'cuda:0\').eval()\n        # whmodel is stored in a plain Python list and is NOT registered as a submodule.\n        # Counting self.model.parameters() alone would miss the entire Whisper medium.\n        self.whisper = self.model.whmodel[0].eval()\n        self.model.decoding_options = whisper.DecodingOptions(\n            language=\'vi\', task=\'transcribe\', temperature=0, beam_size=None,\n            without_timestamps=True, fp16=True, sample_len=224)\n        self.counts = count_modules([self.model, self.whisper])\n        self.provenance = dict(whisper_name=self.model.whisper_model_name,\n            whisper_download_url=whisper._MODELS[self.model.whisper_model_name],\n            language=\'vi\', decoding=\'greedy temperature=0; no prompt/context; sample_len=224\',\n            quantizer_dtype=\'float32\', whisper_dtype=\'float32 weights, fp16 decoding\')\n\n    def decode(self, audio, sr):\n        import torch\n        with torch.inference_mode():\n            output = self.model.inference(torch.from_numpy(audio.copy()).unsqueeze(0).to(\'cuda:0\'))\n        if len(output) != 1:\n            raise ValueError(\'Expected one Ichigo result\')\n        # OpenAI Whisper returns content tokens without an EOS marker. Reaching the\n        # configured maximum is conservatively treated as unfinished, not success.\n        if len(output[0].tokens) >= 224:\n            raise RuntimeError(\'ICHIGO_TOKEN_LIMIT: transcript may be unfinished\')\n        return output[0].text\n\n\nclass Sherpa(Engine):\n    def __init__(self, paths):\n        import onnx\n        import sherpa_onnx\n        components = [\'encoder-epoch-12-avg-8.onnx\', \'decoder-epoch-12-avg-8.onnx\', \'joiner-epoch-12-avg-8.onnx\']\n        # Count every stored initializer, including constants/duplicated exported\n        # tensors. This is an upper bound, not a claimed exact trainable count.\n        total = 0\n        for name in components:\n            graph = onnx.load(str(paths[name]), load_external_data=False)\n            total += onnx_tensor_elements(graph.graph)\n        self.counts = dict(bound=total, basis=\'sum_onnx_stored_tensor_elements_conservative_bound\')\n        self.model = sherpa_onnx.OfflineRecognizer.from_transducer(\n            encoder=str(paths[components[0]]), decoder=str(paths[components[1]]),\n            joiner=str(paths[components[2]]), tokens=str(paths[\'tokens.txt\']),\n            num_threads=2, sample_rate=16000, feature_dim=80,\n            decoding_method=\'greedy_search\', provider=\'cpu\')\n        self.provenance = dict(provider=\'cpu\', threads=2, decoding=\'greedy_search\',\n                               hotwords=False, punctuation_model=False, external_lm=False)\n\n    def decode(self, audio, sr):\n        stream = self.model.create_stream()\n        stream.accept_waveform(sr, audio)\n        self.model.decode_stream(stream)\n        return stream.result.text\n\n\ndef main(request_path, name, engine_factory=None, token=None):\n    from huggingface_hub import HfApi, hf_hub_download\n    req = json.loads(Path(request_path).read_text(encoding=\'utf-8\'))\n    spec = next(s for s in req[\'models\'] if s[\'name\'] == name)\n    if spec[\'family\'] not in (\'ichigo\', \'sherpa\'):\n        raise ValueError(\'Unsupported model family\')\n    out = Path(req[\'output_dir\'])/req[\'stage\']/name\n    out.mkdir(parents=True, exist_ok=True)\n    b.write_json(out/\'state.json\', dict(state=\'starting\'))\n    try:\n        # Immutable revisions are selected in the notebook, never a moving main.\n        info = HfApi(token=token).model_info(spec[\'model_id\'], revision=spec[\'revision\'])\n        paths = {f: Path(hf_hub_download(spec[\'model_id\'], filename=f, revision=info.sha, token=token)) for f in spec[\'files\']}\n        hashes = {f: b.digest_file(p) for f, p in paths.items()}\n        identity = dict(spec=spec, revision=info.sha, file_sha256=hashes,\n            input_fingerprint=req[\'input_fingerprint\'], source_hash=req[\'source_hash\'],\n            worker_hash=b.digest_file(__file__), normalization=b.NORMALIZATION,\n            versions={d.metadata[\'Name\']: d.version for d in metadata.distributions()})\n        config = out/\'config.json\'\n        if config.exists() and json.loads(config.read_text(encoding=\'utf-8\')) != identity:\n            raise ValueError(\'Resume config changed; use a new OUTPUT_DIR to avoid mixing runs\')\n        b.write_json(config, identity)\n        b.write_json(out/\'checkpoint.json\', dict(model_id=spec[\'model_id\'], revision=info.sha, files=hashes))\n        journal = out/\'predictions.jsonl\'\n        records = b.read_journal(journal)\n        inference_rows = req[\'inference_rows\']  # No reference is passed to either engine.\n        if any(r[\'ref_uid\'] not in {x[\'ref_uid\'] for x in inference_rows} for r in records):\n            raise ValueError(\'Foreign prediction IDs\')\n        latest = {r[\'ref_uid\']: r for r in records}\n        pending = [r for r in inference_rows if r[\'ref_uid\'] not in latest or\n                   (req[\'retry_errors\'] and latest[r[\'ref_uid\']][\'status\'] != \'ok\')]\n        if not pending:\n            b.write_json(out/\'state.json\', dict(state=\'completed\', resumed=True))\n            return\n        started = time.perf_counter()\n        engine = engine_factory(spec, paths) if engine_factory is not None else (\n            Ichigo(paths) if spec[\'family\'] == \'ichigo\' else Sherpa(paths))\n        count = engine.counts[\'bound\']\n        b.write_json(out/\'parameters.json\', dict(asr_parameters=count, **engine.counts,\n            below_2B=count < LIMIT, llm_parameters=req[\'llm_parameters\'],\n            system_parameters=count+req[\'llm_parameters\']+req[\'other_parameters\'],\n            below_6B=count+req[\'llm_parameters\']+req[\'other_parameters\'] < 6_000_000_000,\n            note=\'Conservative bound includes all ASR components; do not read as exact trainable count.\'))\n        require_under_limit(count)\n        b.write_json(out/\'runtime.json\', dict(load_ms=(time.perf_counter()-started)*1000,\n            device=\'cuda:0\' if engine.uses_cuda else \'cpu\', **engine.provenance))\n        print(name, \'parameter bound:\', count, \'| pending:\', len(pending), flush=True)\n        valid = next((r for r in pending if not r.get(\'preflight_error\')), None)\n        if valid is None:\n            raise ValueError(\'No valid audio for warmup\')\n        warm = engine.transcribe(valid[\'audio_path\'])\n        b.write_json(out/\'warmup.json\', warm)\n        if warm[\'status\'] != \'ok\':\n            raise RuntimeError(\'Warmup failed: \'+str(warm[\'error_code\']))\n        for i, row in enumerate(pending, 1):\n            result = dict(status=\'error\', text_model=\'\', error_code=\'INPUT_PREFLIGHT: \'+row[\'preflight_error\']) if row.get(\'preflight_error\') else engine.transcribe(row[\'audio_path\'])\n            if result.get(\'duration_s\') is not None and abs(result[\'duration_s\']-row[\'duration\']) > 1/16000:\n                result.update(status=\'error\', error_code=\'CACHE_AUDIO_DURATION_MISMATCH\')\n            b.append_json(journal, dict(ref_uid=row[\'ref_uid\'], **result))\n            if engine.uses_cuda and result[\'status\'] != \'ok\' and \'out of memory\' in str(result.get(\'error_code\')).lower():\n                import torch\n                torch.cuda.empty_cache()\n            if i % 10 == 0 or i == len(pending):\n                print(name, i, \'/\', len(pending), result[\'status\'], flush=True)\n        b.write_json(out/\'state.json\', dict(state=\'completed\', n_selected=len(inference_rows)))\n    except Exception:\n        b.write_json(out/\'state.json\', dict(state=\'failed\', traceback=traceback.format_exc()))\n        raise\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'request\')\n    parser.add_argument(\'model\')\n    args = parser.parse_args()\n    main(args.request, args.model)\n', 'zipformer_vi_worker.py': '"""Vietnamese 153M/291M ONNX adapters using the existing journal/scorer protocol."""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\n\nimport additional_asr_worker as worker\n\n\ndef read_hf_token():\n    try:\n        from kaggle_secrets import UserSecretsClient\n    except ImportError:\n        value = os.environ.get(\'HF_TOKEN\', \'\')\n    else:\n        try:\n            value = UserSecretsClient().get_secret(\'HF_TOKEN\')\n        except Exception:\n            raise RuntimeError(\'Không đọc được Kaggle Secret HF_TOKEN. Thêm secret và bật công tắc cho notebook.\') from None\n    value = str(value or \'\').strip()\n    if not value.startswith(\'hf_\') or any(c.isspace() for c in value):\n        raise RuntimeError(\'Cần token Hugging Face đầy đủ bắt đầu bằng hf_, lưu trong Kaggle Secrets tên HF_TOKEN.\')\n    os.environ[\'HF_TOKEN\'] = value\n    return value\n\n\ndef ensure_access(spec):\n    """Public models use token=False; gated models get an explicit refreshed token."""\n    from huggingface_hub import HfApi, get_hf_file_metadata, hf_hub_url\n    from huggingface_hub.errors import GatedRepoError, HfHubHTTPError\n    token = read_hf_token() if spec.get(\'gated\') else False\n    if spec.get(\'gated\'):\n        try:\n            HfApi(token=token).whoami()\n        except HfHubHTTPError:\n            raise RuntimeError(\'HF_TOKEN không xác thực được. Kiểm tra token mới có quyền đọc và Internet.\') from None\n    try:\n        get_hf_file_metadata(hf_hub_url(spec[\'model_id\'], spec[\'tokens\'], revision=spec[\'revision\']), token=token)\n    except GatedRepoError:\n        raise RuntimeError(\'Chưa có quyền tải \'+spec[\'model_id\']+\'. Mở https://huggingface.co/\'+spec[\'model_id\']+\n            \' bằng đúng tài khoản tạo token, chấp nhận điều kiện truy cập; cấp quyền đọc repo cho token rồi rerun cell model.\') from None\n    except HfHubHTTPError:\n        raise RuntimeError(\'Không đọc được file model \'+spec[\'model_id\']+\'. Kiểm tra quyền tải, Internet và revision.\') from None\n    return token\n\n\ndef read_token_table(path):\n    """Read exact publisher IDs without assuming a particular spelling for blank."""\n    tokens, ids = {}, set()\n    for line in Path(path).read_text(encoding=\'utf-8-sig\').splitlines():\n        if not line.strip():\n            continue\n        symbol, identifier = line.rsplit(maxsplit=1)\n        value = int(identifier)\n        if symbol in tokens or value in ids or value < 0:\n            raise ValueError(\'Duplicate or invalid token IDs in checkpoint vocabulary\')\n        tokens[symbol] = value\n        ids.add(value)\n    if 0 not in ids:\n        raise ValueError(\'Checkpoint token table is missing ID 0; refusing to invent or renumber IDs\')\n    return tokens\n\n\ndef validate_tokens(path, expected_vocab_size=None):\n    """Validate decoder output IDs; allow additional publisher special symbols.\n\n    A publisher can name ID 0 <blank>/<eps> instead of <blk>, or append\n    disambiguation symbols outside the decoder output range. Neither is a reason\n    to renumber/reject an otherwise complete model vocabulary.\n    """\n    tokens = read_token_table(path)\n    ids = set(tokens.values())\n    if expected_vocab_size is None:\n        return len(ids)\n    if expected_vocab_size <= 0:\n        raise ValueError(\'Invalid decoder vocab_size metadata\')\n    missing = set(range(expected_vocab_size))-ids\n    if missing:\n        raise ValueError(f\'Token table missing {len(missing)} decoder output IDs; first missing: {min(missing)}\')\n    return expected_vocab_size\n\n\nclass Zipformer(worker.Engine):\n    def __init__(self, spec, paths):\n        import onnx\n        import sherpa_onnx\n        tokens = read_token_table(paths[spec[\'tokens\']])\n        total, declared_sizes = 0, set()\n        for component in [\'encoder\', \'decoder\', \'joiner\']:\n            graph = onnx.load(str(paths[spec[component]]), load_external_data=False)\n            total += worker.onnx_tensor_elements(graph.graph)\n            metadata = {p.key: p.value for p in graph.metadata_props}\n            if component == \'decoder\' and \'vocab_size\' not in metadata:\n                raise ValueError(\'Decoder ONNX is missing required vocab_size metadata\')\n            if \'vocab_size\' in metadata:\n                declared_sizes.add(int(metadata[\'vocab_size\']))\n        if len(declared_sizes) != 1:\n            raise ValueError(\'Missing or inconsistent ONNX vocab_size metadata; cannot validate output IDs\')\n        vocab_size = validate_tokens(paths[spec[\'tokens\']], declared_sizes.pop())\n        token_zero = next(symbol for symbol, value in tokens.items() if value == 0)\n        print(\'Vocabulary:\', vocab_size, \'| table rows:\', len(tokens), \'| ID 0:\', repr(token_zero),\n              \'| original IDs preserved\', flush=True)\n        # Fail before initializing the recognizer if the stored tensor bound is >=2B.\n        worker.require_under_limit(total)\n        self.counts = dict(bound=total, basis=\'sum_onnx_stored_tensor_elements_conservative_bound\')\n        self.model = sherpa_onnx.OfflineRecognizer.from_transducer(\n            encoder=str(paths[spec[\'encoder\']]), decoder=str(paths[spec[\'decoder\']]),\n            joiner=str(paths[spec[\'joiner\']]), tokens=str(paths[spec[\'tokens\']]),\n            num_threads=2, sample_rate=16000, feature_dim=80,\n            decoding_method=\'greedy_search\', provider=\'cpu\')\n        self.provenance = dict(provider=\'cpu\', threads=2, decoding=\'greedy_search\',\n            precision=\'float32 ONNX files\', hotwords=False, external_lm=False,\n            punctuation_model=False, vocabulary_size=vocab_size,\n            token_source=spec[\'tokens\'], token_id_zero_symbol=token_zero,\n            token_table_rows=len(tokens), ids_renumbered=False,\n            components={k: spec[k] for k in [\'encoder\',\'decoder\',\'joiner\']})\n\n    def decode(self, audio, sr):\n        stream = self.model.create_stream()\n        stream.accept_waveform(sr, audio)\n        self.model.decode_stream(stream)\n        return stream.result.text\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'request\')\n    parser.add_argument(\'model\')\n    args = parser.parse_args()\n    req = json.loads(Path(args.request).read_text(encoding=\'utf-8\'))\n    spec = next(s for s in req[\'models\'] if s[\'name\'] == args.model)\n    token = ensure_access(spec)\n    worker.main(args.request, args.model, engine_factory=Zipformer, token=token)\n'}
for name, source in SOURCES.items():
    (CODE_DIR/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(CODE_DIR))
import asr_benchmark as b
from parallel_benchmark_report import summarize, export_results
SOURCE_HASH = b.fingerprint({name:hashlib.sha256(source.encode()).hexdigest() for name,source in SOURCES.items()})

## 3. Kiểm tra 991 audio
Đọc đúng ID từ cache cũ. Không chia lại dữ liệu; file thiếu/hỏng được ghi trong `input_file_checks.csv`.


In [ ]:
cache_dir, cache_provenance = b.resolve_followup_cache(FOLLOWUP_INPUT,WORK_DIR,CACHE_DIR)
references, joined, benchmark, cache_hashes = b.load_cache(cache_dir)
# Giữ nguyên đúng 991 ID của lượt chạy cũ, không chia hay lấy tập con.
rows = joined.loc[joined.partition.eq('validation')].copy().sort_values('ref_uid').reset_index(drop=True)
if len(rows) != 991:
    raise ValueError(f'Cần đúng 991 audio của bản cũ, nhận được {len(rows)}')
if any(not b.normalize(x) for x in rows.reference):
    raise ValueError('Reference rỗng sau normalization; cần kiểm tra gold trước.')
try:
    AUDIO_ROOT, audio_provenance = b.prepare_selected_audio(rows,DATASET_ROOT,WORK_DIR,DATASET_ZIP)
except FileNotFoundError:
    # Partial extracted mounts: retain missing files as sample errors, never drop their references.
    mounted = [b.resolve_audio(p,DATASET_ROOT).is_file() for p in rows.path]
    if DATASET_ZIP is not None or not DATASET_ROOT.is_dir() or not any(mounted):
        raise
    AUDIO_ROOT = DATASET_ROOT
    audio_provenance = dict(mode='partial_mounted_files',n_present=sum(mounted),n_selected=len(rows))
stage_dir = OUTPUT_DIR/'full_test'
stage_dir.mkdir(parents=True,exist_ok=True)
input_fingerprint = b.fingerprint(dict(cache_hashes=cache_hashes,stage='full_test',
    selected=rows[['ref_uid','sample_uid','kind','reference','duration']].to_dict('records')))
if RESUME_ZIP is not None:
    if any(stage_dir.iterdir()):
        raise RuntimeError('RESUME_ZIP yêu cầu thư mục stage output trống; bỏ RESUME_ZIP nếu đã khôi phục.')
    with zipfile.ZipFile(RESUME_ZIP) as z:
        manifest = json.loads(z.read('EXPORT_MANIFEST.json'))
        if manifest['input_fingerprint']!=input_fingerprint or manifest['stage']!='full_test':
            raise ValueError('ZIP thuộc tập chấm khác.')
        for entry in manifest['files']:
            target = (stage_dir/entry['path']).resolve()
            if not target.is_relative_to(stage_dir.resolve()) or target.suffix not in {'.csv','.json','.jsonl','.txt','.py','.yaml'}:
                raise ValueError('Unsafe resume member')
            content = z.read(entry['path'])
            if hashlib.sha256(content).hexdigest()!=entry['sha256']:
                raise ValueError('Resume ZIP checksum mismatch')
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(content)
    RESUME_ZIP = None
checks, inference_rows = [], []
for row in rows.itertuples():
    path = b.resolve_audio(row.path,AUDIO_ROOT)
    check = dict(ref_uid=row.ref_uid,audio_path=str(path),expected_duration=float(row.duration),status='ok',error='')
    try:
        info = sf.info(path)
        check.update(sample_rate=info.samplerate,channels=info.channels,actual_duration=info.duration)
        if info.samplerate!=16000 or info.channels!=1:
            raise ValueError('Expected mono 16000 Hz')
        if info.frames<=0 or abs(info.duration-float(row.duration))>1/16000:
            raise ValueError('Empty file or cache/audio duration mismatch')
    except Exception as exc:
        check.update(status='error',error=type(exc).__name__+': '+str(exc))
    checks.append(check)
    inference_rows.append(dict(ref_uid=row.ref_uid,audio_path=str(path),duration=float(row.duration),
                               preflight_error=check['error']))
pd.DataFrame(checks).to_csv(stage_dir/'input_file_checks.csv',index=False,encoding='utf-8-sig')
rows.to_csv(stage_dir/'selection_with_references.csv',index=False,encoding='utf-8-sig')
request = dict(output_dir=str(OUTPUT_DIR),stage='full_test',models=MODELS,
    input_fingerprint=input_fingerprint,source_hash=SOURCE_HASH,cache_hashes=cache_hashes,
    llm_parameters=LLM_PARAMETERS,other_parameters=OTHER_PARAMETERS,retry_errors=RETRY_ERRORS,
    inference_rows=inference_rows)
REQUEST_PATH = stage_dir/'request.json'
b.write_json(REQUEST_PATH,request)
b.write_json(stage_dir/'input_provenance.json',dict(cache=cache_provenance,audio=audio_provenance,
    cache_hashes=cache_hashes,input_fingerprint=input_fingerprint,normalization=b.NORMALIZATION))
for name,source in SOURCES.items():
    (stage_dir/name).write_text(source,encoding='utf-8')
print('Tập chạy:','full_test','| audio:',len(rows),'| conversations:',rows.sample_uid.nunique())
print('input_fingerprint:',input_fingerprint)
print('Input lỗi:',sum(x['status']!='ok' for x in checks))
display(pd.DataFrame(checks).query("status != 'ok'"))
(stage_dir/'pip_check.txt').write_text(DEPENDENCY_CHECK.stdout + DEPENDENCY_CHECK.stderr, encoding='utf-8')

## 4. Runtime
Tự cài Python 3.11 và Sherpa-ONNX trên CPU bằng uv, không dùng ensurepip.


In [ ]:
def logged_command(command, log_path):
    print('Đang chạy:', ' '.join(command), flush=True)
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write('COMMAND: ' + repr(command) + '\n')
        log.flush()
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1,
                              env={**os.environ, 'PYTHONUNBUFFERED': '1'}) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise RuntimeError(f'Process thất bại (exit {result}); xem {log_path}')

from zipformer_vi_worker import ensure_access
REQUIREMENTS = ['numpy==1.26.4','pandas==2.2.3','soundfile==0.13.1','psutil==7.1.0',
                'sherpa-onnx==2.0.1','onnx==1.23.2','huggingface-hub==1.24.0']
os.environ['UV_CACHE_DIR'] = '/kaggle/working/lifemate_zipformer_uv_cache'
os.environ['UV_PYTHON_INSTALL_DIR'] = '/kaggle/working/lifemate_zipformer_python'

def ensure_environment(name):
    directory = stage_dir/name
    directory.mkdir(exist_ok=True)
    env = WORK_DIR/'sherpa_environment'
    python = env/'bin/python'
    marker = env/'requirements_ready.json'
    if not python.exists():
        logged_command([sys.executable,'-m','uv','venv','--python','3.11',
                        '--managed-python',str(env)], directory/'install.log.txt')
    if not marker.exists() or json.loads(marker.read_text()) != REQUIREMENTS:
        logged_command([sys.executable,'-m','uv','pip','install','--python',
                        str(python),*REQUIREMENTS], directory/'install.log.txt')
    probe = (
        'import sys,json,sherpa_onnx,onnx,soundfile,psutil; '
        'import importlib.metadata as md; from pathlib import Path; '
        'print("Sherpa CPU 2 threads; no GPU required"); '
        'Path(sys.argv[1]).write_text(json.dumps(dict(python=sys.version,executable=sys.executable, '
        'versions={d.metadata["Name"]:d.version for d in md.distributions()}),indent=2))')
    logged_command([str(python),'-c',probe,str(directory/'environment.json')],
                   directory/'runtime_check.log.txt')
    marker.write_text(json.dumps(REQUIREMENTS),encoding='utf-8')
    return python

def refresh_results():
    summary = summarize(rows, request)
    display(summary.loc[summary.subset.eq('all'), [
        'model', 'n', 'n_ok', 'n_error', 'n_missing', 'WER_pct', 'CER_pct',
        'exact_normalized_pct', 'comparison_ready', 'state']])
    archive = export_results(request)
    display(FileLink(os.path.relpath(archive, Path.cwd())))
    return summary

def run_model(name):
    directory = stage_dir/name
    directory.mkdir(exist_ok=True)
    print('MODEL:', name, '| Chạy toàn bộ', len(rows), 'audio', flush=True)
    try:
        spec = next(s for s in MODELS if s['name'] == name)
        python = ensure_environment(name)
        logged_command([str(python), '-u', str(CODE_DIR/'zipformer_vi_worker.py'),
                        str(REQUEST_PATH), name], directory/'worker.log.txt')
    except Exception:
        message = traceback.format_exc()
        state_path = directory/'state.json'
        previous = json.loads(state_path.read_text()) if state_path.exists() else {}
        b.write_json(state_path, dict(state='failed', traceback=message, worker_state=previous))
        print(message)
    finally:
        refresh_results()

## 5. hynt ZipFormer-150M — khoảng 153M
FP32 ONNX epoch-11-avg-2, CPU 2 threads, greedy. Bảng token nằm trong config.json, không parse file đó như JSON.


In [ ]:
run_model('zipformer_hynt_153m')


## Kết quả và ZIP
`metrics.csv`: WER/CER %, exact match, số OK/error/missing; `per_audio_all_models.csv`: reference/prediction từng audio.
`errors_and_missing.csv` và `<model>/worker.log.txt` ghi lỗi cụ thể. ZIP được tạo sau mỗi model.
`comparison_ready=True` khi đủ 991 prediction thành công. Lỗi/thiếu vẫn giữ trong mẫu số;
chưa có mẫu thành công thì không có WER/CER hợp lệ.
991 audio giữ nguyên từ tập đánh giá cũ, không phải bộ test độc lập mới.
Phiên mới muốn tiếp tục: Add Input ZIP output rồi điền `RESUME_ZIP`. Giữ output private.


In [ ]:
summary = refresh_results()
display(pd.read_csv(stage_dir/'error_counts.csv').head(30))


Nguồn: [hynt model card](https://huggingface.co/hynt/ZipFormer-150M-CR-CTC-RNNT-6000h).
Quy mô khoảng 153M theo card, bound ONNX đã đo 149.906.361 scalar, không phải số trainable chính xác.
Run đã có WER 15,34%, CER 10,65%; đây là pretrained trên development validation cũ,
không phải cải thiện do fine-tune hoặc test độc lập. License CC-BY-NC-ND-4.0 theo card.
